# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print overview
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}\n")
print("Authors: ")
for author in getattr(dataset.metadata, 'author', []):
    print(f"  - {author.get('name') if isinstance(author, dict) else author}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** All further operations use Croissant entity `@id` fields for clarity and reproducibility.

In [ ]:
# List all record sets with their IDs and their fields' IDs and names.
print("Available Record Sets:")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"- Record Set Name: {rs.name} (@id={rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}, type={field.data_type})")
    record_sets.append(rs.id)

if not record_sets:
    print("No explicit record sets found in `record_sets`. Using dataset-distributed record sets.")
    # Try to parse from files if not directly present
    for rso in dataset.metadata._all_record_sets_and_files():
        # Shows both RecordSets and FileObject (some datasets emit FileObject, pointing to table)
        print(f"- Record Set Name: {rso.name} (@id={rso.id})")
        if hasattr(rso, 'fields'):
            print("  Fields:")
            for field in getattr(rso, 'fields', []):
                print(f"    - {field.name} (@id={field.id}, type={field.data_type})")
        record_sets.append(rso.id)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If you saw available record set IDs above, set them here. For this dataset, there's likely only one main record set.
# Fill out the main record set ID you see (example; replace with actual from print above):
main_record_set_id = None
for rs in dataset.metadata._all_record_sets_and_files():
    # Choose the first tabular set for demo:
    main_record_set_id = rs.id
    break
if not main_record_set_id:
    raise RuntimeError("No record set found!")

print(f"Using record set: {main_record_set_id}")

# Load all records from that record set:
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded {len(df)} records.\n")
print("Columns:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping data by key attributes. All field selections use their `@id` strings.

Typical numeric/quantitative fields in proteomics include peptide counts and molecular weight (MW).

In [ ]:
# Find suitable numeric fields (change as per your dataset):
for col in df.columns:
    # Print a sample with unique values for inspection
    print(f"Field: {col}, Sample values: {df[col].dropna().unique()[:5]}")

# Select a numeric field by its @id (example: 'peptide_count', update to the real @id from previous cell's output)
numeric_field = None
for col in df.columns:
    if (df[col].dtype==float or df[col].dtype==int or np.issubdtype(df[col].dtype, np.number)):
        numeric_field = col
        break
if not numeric_field:
    # Try to convert likely numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field = col
            break
        except Exception:
            continue
if not numeric_field:
    print("No numeric field found.")
else:
    print(f"Using {numeric_field} as numeric field.")

threshold = (df[numeric_field].mean() + df[numeric_field].std()) if numeric_field else None
if numeric_field:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (for example: sample or protein @id)
group_field = None
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() < 20 and col != numeric_field:
        group_field = col
        break

if group_field and numeric_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and if available, compare groups by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- We loaded and inspected the FAIR\u2072 dataset for mass spectrometry proteins.
- The main record set and its fields were discovered and referenced by `@id`.
- We filtered, normalized, and visualized quantitative features (e.g., peptide counts or MW).
- All data manipulation referenced Croissant entities only by their unique `@id` identifiers.

Further analyses could focus on correlating protein abundances, peptide identifications, and possible sample condition effects, depending on the full structure of the dataset's record sets and fields.

Refer to the [FAIR² dataset page](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for in-depth schema and documentation.